# 第10章 Section 10.2: 数组与排序搜索 (Arrays, Sorting & Searching)
## Cambridge International AS & A Level Computer Science (9618)

---

> **核心问题**: 当我们需要存储大量相同类型的数据时, 为每个值创建一个变量显然不现实. 我们需要一种结构来 "批量管理" 数据 --- 这就是 **数组 (Array)**.

在这一节中, 我们将学习数组的概念、一维和二维数组, 以及两个经典算法: **冒泡排序 (Bubble Sort)** 和 **线性搜索 (Linear Search)**.

## 1. 什么是数组? (What is an Array?)

### 定义
**数组 (Array)** 是一种数据结构, 它在内存中存储 **固定数量** 的 **相同类型** 元素, 并通过 **索引 (index)** 访问每个元素.

### 现实世界的类比: 信箱

想象公寓楼的一排信箱:
```
+------+------+------+------+------+------+
| 信件A | 信件B | 信件C | 信件D | 信件E | 信件F |
+------+------+------+------+------+------+
  [0]    [1]    [2]    [3]    [4]    [5]
```

- 每个信箱 **大小相同** (存储相同类型的数据)
- 每个信箱有 **编号** (索引 index)
- 你可以直接走到 3 号信箱取信 --- 不需要从头开始找!

### 为什么随机访问是 O(1)?

因为数组在内存中是 **连续存储 (contiguous)** 的:
```
内存地址:  1000  1004  1008  1012  1016  1020
数组元素:  [10]  [20]  [30]  [40]  [50]  [60]
索引:       [0]   [1]   [2]   [3]   [4]   [5]
```

要访问 `array[3]`, 计算机只需: `起始地址 + 3 x 元素大小 = 1000 + 3 x 4 = 1012`
直接跳到那个地址! 不管数组有多大, 访问任何元素都一样快.

### 关键术语
| 术语 | 解释 |
|------|------|
| **Index (索引)** | 元素的位置编号 |
| **Lower bound (下界)** | 最小的索引值 (pseudocode 中通常是 0 或 1) |
| **Upper bound (上界)** | 最大的索引值 |
| **Size / Length** | 元素总数 = 上界 - 下界 + 1 |

In [ ]:
%matplotlib inline

# === 中文字体配置 (Chinese Font Setup) ===
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')

# 方法1: 全局设置 SimHei 字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DengXian', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 方法2: 强制重建字体缓存（首次运行可能需要）
fm._load_fontmanager(try_read_cache=False)

# 验证
test_font = fm.findfont('SimHei')
if 'SimHei' in test_font or 'simhei' in test_font.lower():
    print('✅ 中文字体 SimHei 已加载')
else:
    print(f'⚠️ SimHei 未找到，使用: {test_font}')
    # Fallback: 直接注册字体文件
    font_path = 'C:/Windows/Fonts/simhei.ttf'
    if __import__('os').path.exists(font_path):
        fm.fontManager.addfont(font_path)
        plt.rcParams['font.sans-serif'] = ['SimHei'] + plt.rcParams['font.sans-serif']
        print('✅ 已手动加载 SimHei 字体文件')

import matplotlib.patches as patches
import numpy as np

# 可视化: 数组在内存中的布局
fig, ax = plt.subplots(1, 1, figsize=(12, 3))

arr = [15, 42, 8, 23, 56, 31]
n = len(arr)

for i in range(n):
    # 画格子
    rect = patches.FancyBboxPatch((i * 1.8, 0.5), 1.5, 1.2,
                                   boxstyle="round,pad=0.05",
                                   facecolor='#4ECDC4', edgecolor='#2C3E50', linewidth=2)
    ax.add_patch(rect)
    # 值
    ax.text(i * 1.8 + 0.75, 1.1, str(arr[i]), ha='center', va='center',
            fontsize=16, fontweight='bold', color='white')
    # 索引
    ax.text(i * 1.8 + 0.75, 0.15, f'[{i}]', ha='center', va='center',
            fontsize=12, color='#2C3E50')
    # 内存地址
    ax.text(i * 1.8 + 0.75, 2.0, f'{1000 + i * 4}', ha='center', va='center',
            fontsize=9, color='#7F8C8D')

ax.set_xlim(-0.3, n * 1.8 + 0.3)
ax.set_ylim(-0.3, 2.5)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Array in Memory (数组在内存中的布局)', fontsize=14, fontweight='bold', pad=10)
ax.text(-0.2, 2.0, '内存地址:', fontsize=9, color='#7F8C8D', ha='right')
ax.text(-0.2, 0.15, '索引:', fontsize=10, color='#2C3E50', ha='right')
plt.tight_layout()
plt.show()


## 2. 一维数组 (1D Array)

### Pseudocode 声明
```
// 声明一个包含 6 个整数的数组 (索引 0 到 5)
DECLARE Numbers : ARRAY[0:5] OF INTEGER

// 赋值
Numbers[0] <- 15
Numbers[1] <- 42
Numbers[2] <- 8

// 读取
OUTPUT Numbers[0]    // 输出 15
```

### Python 实现
在 Python 中, 我们用 **列表 (list)** 来实现数组的功能 (虽然 list 更灵活).

In [ ]:
# 1D 数组基本操作
# 声明和初始化
numbers = [15, 42, 8, 23, 56, 31]
print(f"数组: {numbers}")
print(f"长度: {len(numbers)}")
print(f"下界: 0, 上界: {len(numbers) - 1}")
print()

# 访问元素
print(f"numbers[0] = {numbers[0]}  (第一个元素)")
print(f"numbers[3] = {numbers[3]}  (第四个元素)")
print(f"numbers[5] = {numbers[5]}  (最后一个元素)")
print()

# 修改元素
numbers[2] = 99
print(f"修改 numbers[2] = 99 后: {numbers}")
print()

# 遍历数组 (Traversal)
print("=== 遍历数组 ===")
print("方法1: 用索引遍历")
for i in range(len(numbers)):
    print(f"  numbers[{i}] = {numbers[i]}")

print()
print("方法2: 直接遍历值")
for value in numbers:
    print(f"  值: {value}")

In [ ]:
# 1D 数组常见操作
scores = [78, 92, 65, 88, 71, 95, 83]

# 求总和与平均值
total = 0
for score in scores:
    total += score
average = total / len(scores)
print(f"成绩: {scores}")
print(f"总分: {total}")
print(f"平均分: {average:.1f}")
print()

# 查找最大值和最小值
max_val = scores[0]
min_val = scores[0]
for i in range(1, len(scores)):
    if scores[i] > max_val:
        max_val = scores[i]
    if scores[i] < min_val:
        min_val = scores[i]
print(f"最高分: {max_val}")
print(f"最低分: {min_val}")
print()

# 计数: 有多少个分数 >= 80?
count = 0
for score in scores:
    if score >= 80:
        count += 1
print(f"80分及以上: {count} 人")

## 3. 二维数组 (2D Array)

### 什么是 2D 数组?

二维数组就像一个 **表格 (table)** 或 **电子表格 (spreadsheet)**:
- **行 (rows)** 和 **列 (columns)**
- 用两个索引访问: `Array[row][col]`

### Pseudocode 声明
```
// 3 行 4 列的二维数组
DECLARE Grid : ARRAY[0:2, 0:3] OF INTEGER

// 赋值
Grid[0][0] <- 1
Grid[0][1] <- 2
Grid[1][0] <- 5

// 遍历 (嵌套循环)
FOR Row <- 0 TO 2
    FOR Col <- 0 TO 3
        OUTPUT Grid[Row][Col]
    NEXT Col
NEXT Row
```

### 现实世界的类比: 电影院座位
```
        列0  列1  列2  列3  列4
行0  [ A1 ][ A2 ][ A3 ][ A4 ][ A5 ]
行1  [ B1 ][ B2 ][ B3 ][ B4 ][ B5 ]
行2  [ C1 ][ C2 ][ C3 ][ C4 ][ C5 ]
```
seat[1][3] = B4 号座位

In [ ]:
# 2D 数组基本操作
# 创建 3x4 的二维数组
grid = [
    [1,  2,  3,  4],
    [5,  6,  7,  8],
    [9, 10, 11, 12]
]

print("=== 二维数组 ===")
for row in grid:
    print(row)
print()

# 访问元素
print(f"grid[0][0] = {grid[0][0]}  (第0行, 第0列)")
print(f"grid[1][2] = {grid[1][2]}  (第1行, 第2列)")
print(f"grid[2][3] = {grid[2][3]}  (第2行, 第3列)")
print()

# 嵌套循环遍历
print("=== 用嵌套循环遍历 ===")
rows = len(grid)
cols = len(grid[0])
print(f"行数: {rows}, 列数: {cols}")
print()
for r in range(rows):
    for c in range(cols):
        print(f"  grid[{r}][{c}] = {grid[r][c]:2d}", end="")
    print()  # 换行

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 可视化: 2D 数组彩色网格
grid = [
    [1,  2,  3,  4],
    [5,  6,  7,  8],
    [9, 10, 11, 12]
]

fig, ax = plt.subplots(figsize=(8, 5))
arr = np.array(grid)
cmap = plt.cm.YlOrRd

im = ax.imshow(arr, cmap=cmap, aspect='equal')

# 添加数值标签
for i in range(arr.shape[0]):
    for j in range(arr.shape[1]):
        val = arr[i, j]
        color = 'white' if val > 8 else 'black'
        ax.text(j, i, str(val), ha='center', va='center',
                fontsize=18, fontweight='bold', color=color)

# 标注行列
ax.set_xticks(range(arr.shape[1]))
ax.set_yticks(range(arr.shape[0]))
ax.set_xticklabels([f'列 {j}' for j in range(arr.shape[1])], fontsize=12)
ax.set_yticklabels([f'行 {i}' for i in range(arr.shape[0])], fontsize=12)
ax.set_title('2D Array Visualization (二维数组可视化)', fontsize=14, fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.8, label='Value')
plt.tight_layout()
plt.show()

## 4. 冒泡排序 (Bubble Sort)

### 算法思想

冒泡排序是最简单的排序算法之一. 它的名字来源于: 较大的元素像 **气泡 (bubble)** 一样逐渐 "浮" 到数组的末端.

### 算法步骤
1. 从头开始, 比较相邻的两个元素
2. 如果前一个比后一个大, 就 **交换 (swap)** 它们
3. 继续比较下一对, 直到末尾 --- 这叫一 "趟 (pass)"
4. 重复以上步骤, 每趟排好一个最大值
5. 如果某一趟没有发生交换, 说明已经有序, 提前结束

### Pseudocode
```
PROCEDURE BubbleSort(arr, n)
    FOR i <- 0 TO n - 2
        swapped <- FALSE
        FOR j <- 0 TO n - 2 - i
            IF arr[j] > arr[j + 1] THEN
                // Swap
                temp <- arr[j]
                arr[j] <- arr[j + 1]
                arr[j + 1] <- temp
                swapped <- TRUE
            ENDIF
        NEXT j
        IF NOT swapped THEN
            BREAK   // 已经有序, 提前退出
        ENDIF
    NEXT i
ENDPROCEDURE
```

### 时间复杂度 (Time Complexity)
| 情况 | 复杂度 | 解释 |
|------|--------|------|
| **最好 (Best)** | O(n) | 数组已经有序, 一趟就完成 |
| **最坏 (Worst)** | O(n^2) | 数组完全逆序 |
| **平均 (Average)** | O(n^2) | 一般情况 |

**O(n^2) 是什么意思?** 如果数组有 n 个元素, 最坏情况下需要约 n x n 次比较. 当 n = 1000 时, 需要约 1,000,000 次比较! 所以冒泡排序只适合 **小规模数据**.

### 优缺点
- **优点**: 简单易懂, 代码短, 稳定排序, 能提前终止
- **缺点**: O(n^2) 效率低, 不适合大数据集

In [ ]:
# 冒泡排序实现 (带详细输出)
def bubble_sort_verbose(arr):
    n = len(arr)
    arr = arr.copy()  # 不修改原数组
    print(f"原始数组: {arr}")
    print("-" * 50)

    for i in range(n - 1):
        swapped = False
        print(f"\n第 {i + 1} 趟 (Pass {i + 1}):")

        for j in range(n - 1 - i):
            if arr[j] > arr[j + 1]:
                arr[j], arr[j + 1] = arr[j + 1], arr[j]
                swapped = True
                print(f"  比较 arr[{j}]={arr[j+1]} > arr[{j+1}]={arr[j]} -> 交换! -> {arr}")
            else:
                print(f"  比较 arr[{j}]={arr[j]} <= arr[{j+1}]={arr[j+1]} -> 不交换")

        if not swapped:
            print("  没有交换发生, 数组已有序! 提前结束.")
            break

    print(f"\n排序结果: {arr}")
    return arr

# 测试
data = [64, 34, 25, 12, 22, 11, 90]
bubble_sort_verbose(data)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import numpy as np

# 冒泡排序可视化 (静态步骤展示)
def bubble_sort_steps(arr):
    arr = arr.copy()
    steps = [arr.copy()]
    highlights = [(-1, -1)]
    n = len(arr)
    for i in range(n - 1):
        for j in range(n - 1 - i):
            if arr[j] > arr[j + 1]:
                arr[j], arr[j + 1] = arr[j + 1], arr[j]
                steps.append(arr.copy())
                highlights.append((j, j + 1))
    return steps, highlights

data = [64, 34, 25, 12, 22, 11, 90]
steps, highlights = bubble_sort_steps(data)

# 显示关键步骤
n_show = min(8, len(steps))
indices = np.linspace(0, len(steps) - 1, n_show, dtype=int)

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
colors_default = '#3498DB'
colors_swap = '#E74C3C'
colors_sorted = '#2ECC71'

for idx, ax in zip(indices, axes.flat):
    arr_state = steps[idx]
    hi = highlights[idx]
    colors = []
    for k in range(len(arr_state)):
        if k == hi[0] or k == hi[1]:
            colors.append(colors_swap)
        elif idx == len(steps) - 1:
            colors.append(colors_sorted)
        else:
            colors.append(colors_default)

    bars = ax.bar(range(len(arr_state)), arr_state, color=colors, edgecolor='white', linewidth=1.5)
    for k, v in enumerate(arr_state):
        ax.text(k, v + 1, str(v), ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_ylim(0, max(data) + 15)
    ax.set_title(f'Step {idx}', fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle('Bubble Sort Visualization (冒泡排序可视化)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 简洁版冒泡排序 (用于实际使用)
def bubble_sort(arr):
    n = len(arr)
    for i in range(n - 1):
        swapped = False
        for j in range(n - 1 - i):
            if arr[j] > arr[j + 1]:
                arr[j], arr[j + 1] = arr[j + 1], arr[j]
                swapped = True
        if not swapped:
            break
    return arr

# 测试
test_data = [64, 34, 25, 12, 22, 11, 90]
print(f"排序前: {test_data}")
sorted_data = bubble_sort(test_data.copy())
print(f"排序后: {sorted_data}")

## 5. 线性搜索 (Linear Search)

### 算法思想

线性搜索是最简单的搜索算法: 从第一个元素开始, **逐个检查**, 直到找到目标或到达末尾.

### 现实类比: 在书架上找一本书
想象你面前有一排没有按顺序排列的书. 要找一本特定的书, 你只能从左到右一本一本地看, 直到找到为止.

### Pseudocode
```
FUNCTION LinearSearch(arr, n, target) RETURNS INTEGER
    FOR i <- 0 TO n - 1
        IF arr[i] = target THEN
            RETURN i     // 找到了, 返回索引
        ENDIF
    NEXT i
    RETURN -1            // 没找到, 返回 -1
ENDFUNCTION
```

### 时间复杂度
| 情况 | 复杂度 | 解释 |
|------|--------|------|
| **最好 (Best)** | O(1) | 目标就是第一个元素 |
| **最坏 (Worst)** | O(n) | 目标在最后, 或不存在 |
| **平均 (Average)** | O(n) | 平均检查 n/2 个元素 |

### 特点
- **优点**: 简单, 无需排序, 适用于任何数据
- **缺点**: 对于大数组效率低
- **适用场景**: 数据量小或未排序的数据

In [ ]:
# 线性搜索实现 (带详细输出)
def linear_search_verbose(arr, target):
    print(f"在 {arr} 中搜索 {target}")
    print("-" * 50)
    for i in range(len(arr)):
        print(f"  检查 arr[{i}] = {arr[i]}", end="")
        if arr[i] == target:
            print(f"  -> 找到了! 返回索引 {i}")
            return i
        else:
            print(f"  -> 不匹配, 继续...")
    print(f"  搜索完毕, 未找到 {target}. 返回 -1")
    return -1

# 测试
data = [15, 42, 8, 23, 56, 31, 77]
print("=== 搜索存在的值 ===")
result = linear_search_verbose(data, 23)
print()
print("=== 搜索不存在的值 ===")
result = linear_search_verbose(data, 99)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# 线性搜索可视化
def visualize_linear_search(arr, target):
    found_idx = -1
    for i in range(len(arr)):
        if arr[i] == target:
            found_idx = i
            break

    fig, ax = plt.subplots(figsize=(14, 3))
    n = len(arr)

    for i in range(n):
        if i < found_idx:
            color = '#E74C3C'   # 已检查, 不匹配 (红)
            label = 'checked'
        elif i == found_idx:
            color = '#2ECC71'   # 找到! (绿)
            label = 'FOUND!'
        else:
            color = '#BDC3C7'   # 未检查 (灰)
            label = ''

        rect = patches.FancyBboxPatch((i * 1.8, 0.5), 1.5, 1.2,
                                       boxstyle="round,pad=0.05",
                                       facecolor=color, edgecolor='#2C3E50', linewidth=2)
        ax.add_patch(rect)
        ax.text(i * 1.8 + 0.75, 1.1, str(arr[i]), ha='center', va='center',
                fontsize=14, fontweight='bold', color='white')
        ax.text(i * 1.8 + 0.75, 0.15, f'[{i}]', ha='center', va='center',
                fontsize=10, color='#2C3E50')

    # 搜索箭头
    if found_idx >= 0:
        ax.annotate('TARGET!', xy=(found_idx * 1.8 + 0.75, 1.9),
                    fontsize=12, fontweight='bold', color='#2ECC71',
                    ha='center')

    ax.set_xlim(-0.5, n * 1.8 + 0.5)
    ax.set_ylim(-0.3, 2.5)
    ax.set_aspect('equal')
    ax.axis('off')

    title = f'Linear Search for {target}'
    if found_idx >= 0:
        title += f' - Found at index {found_idx}!'
    else:
        title += ' - Not found!'
    ax.set_title(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

data = [15, 42, 8, 23, 56, 31, 77]
visualize_linear_search(data, 23)
visualize_linear_search(data, 99)

In [ ]:
# 简洁版线性搜索
def linear_search(arr, target):
    for i in range(len(arr)):
        if arr[i] == target:
            return i
    return -1

# 测试
data = [15, 42, 8, 23, 56, 31, 77]
print(f"搜索 23: 索引 = {linear_search(data, 23)}")
print(f"搜索 99: 索引 = {linear_search(data, 99)}")

## 6. 排序 vs 搜索: 何时使用什么?

| 场景 | 推荐方法 | 原因 |
|------|---------|------|
| 数据量小 (< 100) | 线性搜索 | 简单, 开销小 |
| 数据未排序, 只搜一次 | 线性搜索 | 排序后再搜不划算 |
| 数据需要多次搜索 | 先排序, 再用二分搜索 | 排序一次, 多次受益 |
| 几乎已排序的数据 | 冒泡排序 | O(n) 最优情况, 简单实现 |
| 大数据集排序 | 更高级算法 (快排等) | O(n log n) 远优于 O(n^2) |

> **额外知识**: 在 A Level 进阶内容中, 你会学到更多排序算法 (如 Insertion Sort, Merge Sort) 和搜索算法 (如 Binary Search). Binary Search 的时间复杂度是 O(log n), 比 Linear Search 的 O(n) 快得多!

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 时间复杂度对比可视化
n_values = np.arange(1, 101)

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(n_values, n_values, label='O(n) - Linear Search', color='#3498DB', linewidth=2)
ax.plot(n_values, n_values ** 2, label='O(n$^2$) - Bubble Sort (worst)', color='#E74C3C', linewidth=2)
ax.plot(n_values, n_values * np.log2(n_values + 1), label='O(n log n) - Quick Sort', color='#2ECC71', linewidth=2, linestyle='--')
ax.plot(n_values, np.log2(n_values + 1), label='O(log n) - Binary Search', color='#9B59B6', linewidth=2)

ax.set_xlabel('输入大小 n', fontsize=12)
ax.set_ylabel('操作次数', fontsize=12)
ax.set_title('Time Complexity Comparison (时间复杂度对比)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim(0, 2000)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("O(n^2) 增长得多快! 当 n=100 时, O(n^2)=10000, 而 O(n log n) 只有约 665.")

## 7. 互动练习: 冒泡排序步骤模拟器 (Interactive Bubble Sort)

运行下面的代码, 输入你自己的数组, 观察冒泡排序的每一步!

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, clear_output
import time

def bubble_sort_interactive(arr):
    # 冒泡排序交互式可视化
    arr = arr.copy()
    n = len(arr)
    all_states = [(arr.copy(), -1, -1, "初始状态")]

    for i in range(n - 1):
        swapped = False
        for j in range(n - 1 - i):
            if arr[j] > arr[j + 1]:
                arr[j], arr[j + 1] = arr[j + 1], arr[j]
                swapped = True
                all_states.append((arr.copy(), j, j + 1, f"Pass {i+1}: swap arr[{j}] and arr[{j+1}]"))
        all_states.append((arr.copy(), -1, n - 1 - i, f"Pass {i+1} complete - arr[{n-1-i}] is sorted"))
        if not swapped:
            all_states.append((arr.copy(), -1, -1, "No swaps needed - array is sorted!"))
            break

    # 画出所有关键步骤
    n_show = min(12, len(all_states))
    indices = np.linspace(0, len(all_states) - 1, n_show, dtype=int)
    cols = 4
    rows_needed = (n_show + cols - 1) // cols

    fig, axes = plt.subplots(rows_needed, cols, figsize=(16, 3.5 * rows_needed))
    if rows_needed == 1:
        axes = [axes]
    axes_flat = [ax for row in axes for ax in (row if hasattr(row, '__len__') else [row])]

    for plot_idx, state_idx in enumerate(indices):
        ax = axes_flat[plot_idx]
        state, h1, h2, desc = all_states[state_idx]
        colors = []
        for k in range(len(state)):
            if k == h1:
                colors.append('#E74C3C')
            elif k == h2:
                colors.append('#F39C12')
            else:
                colors.append('#3498DB')
        ax.bar(range(len(state)), state, color=colors, edgecolor='white', linewidth=1.2)
        for k, v in enumerate(state):
            ax.text(k, v + 0.5, str(v), ha='center', fontsize=9, fontweight='bold')
        ax.set_title(desc, fontsize=9)
        ax.set_ylim(0, max(arr) + 10)
        ax.set_xticks(range(len(state)))
        ax.set_yticks([])

    for plot_idx in range(n_show, len(axes_flat)):
        axes_flat[plot_idx].axis('off')

    fig.suptitle('Bubble Sort Step-by-Step (冒泡排序逐步演示)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# 运行!
test_array = [38, 27, 43, 3, 9, 82, 10]
print(f"排序数组: {test_array}")
bubble_sort_interactive(test_array)

## 8. 2D 数组实际应用: 学生成绩表

In [ ]:
# 2D 数组应用: 学生成绩表
import matplotlib.pyplot as plt
import numpy as np

# 5个学生, 4门科目的成绩
students = ["Alice", "Bob", "Carol", "David", "Eva"]
subjects = ["Math", "English", "Physics", "CS"]

grades = [
    [92, 85, 88, 95],   # Alice
    [78, 90, 72, 88],   # Bob
    [85, 78, 95, 91],   # Carol
    [90, 82, 80, 77],   # David
    [88, 95, 86, 93],   # Eva
]

# 打印成绩表
print(f"{'学生':<10}", end="")
for subj in subjects:
    print(f"{subj:>10}", end="")
print(f"{'平均分':>10}")
print("-" * 60)

for i in range(len(students)):
    print(f"{students[i]:<10}", end="")
    for j in range(len(subjects)):
        print(f"{grades[i][j]:>10}", end="")
    avg = sum(grades[i]) / len(grades[i])
    print(f"{avg:>10.1f}")

# 可视化
fig, ax = plt.subplots(figsize=(10, 6))
arr = np.array(grades)
im = ax.imshow(arr, cmap='RdYlGn', aspect='auto', vmin=60, vmax=100)

for i in range(arr.shape[0]):
    for j in range(arr.shape[1]):
        ax.text(j, i, str(arr[i, j]), ha='center', va='center',
                fontsize=14, fontweight='bold')

ax.set_xticks(range(len(subjects)))
ax.set_yticks(range(len(students)))
ax.set_xticklabels(subjects, fontsize=12)
ax.set_yticklabels(students, fontsize=12)
ax.set_title('Student Grades Heatmap (学生成绩热力图)', fontsize=14, fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.8, label='Score')
plt.tight_layout()
plt.show()

## 9. 练习题 (Practice Exercises)

### 练习 1: 数组操作
给定数组 `data = [45, 12, 78, 34, 56, 89, 23]`:
1. 求所有元素的总和
2. 找出最大值及其索引
3. 将所有偶数元素替换为 0
4. 将数组反转

### 练习 2: 手动冒泡排序
对数组 `[5, 3, 8, 1, 2]` 手动执行冒泡排序, 写出每一趟后的数组状态.

### 练习 3: 2D 数组
创建一个 4x4 的乘法表 (multiplication table), 其中 `table[i][j] = (i+1) * (j+1)`.

### 练习 4: 搜索练习
编写一个函数, 在二维数组中搜索一个值, 返回它的 (行, 列) 位置.

In [ ]:
# 练习参考答案

# 练习 1
data = [45, 12, 78, 34, 56, 89, 23]
print("=== 练习 1 ===")
print(f"1. 总和: {sum(data)}")

max_val = data[0]
max_idx = 0
for i in range(1, len(data)):
    if data[i] > max_val:
        max_val = data[i]
        max_idx = i
print(f"2. 最大值: {max_val}, 索引: {max_idx}")

data_copy = data.copy()
for i in range(len(data_copy)):
    if data_copy[i] % 2 == 0:
        data_copy[i] = 0
print(f"3. 偶数替换为0: {data_copy}")

reversed_data = data[::-1]
print(f"4. 反转: {reversed_data}")
print()

# 练习 3: 乘法表
print("=== 练习 3: 乘法表 ===")
table = []
for i in range(4):
    row = []
    for j in range(4):
        row.append((i + 1) * (j + 1))
    table.append(row)

for row in table:
    for val in row:
        print(f"{val:4d}", end="")
    print()
print()

# 练习 4: 2D 搜索
print("=== 练习 4: 2D 搜索 ===")
def search_2d(grid, target):
    for r in range(len(grid)):
        for c in range(len(grid[r])):
            if grid[r][c] == target:
                return (r, c)
    return (-1, -1)

result = search_2d(table, 6)
print(f"在乘法表中搜索 6: 位置 = {result}")

---
## 本节要点回顾

1. **数组** 是存储相同类型元素的有序集合, 通过索引 O(1) 随机访问
2. **1D 数组**: 线性结构, 用单个索引访问
3. **2D 数组**: 表格结构, 用行和列两个索引访问, 嵌套循环遍历
4. **冒泡排序**: 相邻比较交换, O(n^2) 时间, 简单但慢
5. **线性搜索**: 逐个检查, O(n) 时间, 适用于未排序数据
6. 选择算法要考虑: 数据量、是否已排序、搜索频率

> **下一节**: 文件操作 (Files) 和抽象数据类型 (ADTs) --- Stack, Queue, Linked List!